# 3.6: Generalization

As machine learning scientists, our goal is to discover patterns. 

Overfitting: fitting closer to our training data than the underlying distribution. Often combatted by regularization (and if needed, statistical knowledge).

## 3.6.1: Training Error and Generalization Error

In the standard supervised learning setting, we assume that the training data and test data are drawn *independently* from *identical* distributions. This is the *IID (Independent and Identically Distributed) assumption*. This means we must drastically rethink our strategy if the assumption no longer applies.

Why should we believe that training data sampled from distribution $P(X,Y)$ should tell us how to make predictions on test data generated by a different distributino $Q(X,Y)$? First, we need to understand the IID case, where $P(\cdot) = Q(\cdot)$. Later, we will discuss some assumptions that allow for shifts in distribution.

- $R_{emp}$: training error, statistic calculated from training dataset
- $R$: generalization error, expectation taken WRT underlying dstribution. Can be thought of what we would see if we applied the model to an infinite stream of additional data examples drawn from the underlying data distribution.

Formally, training error is defined as 
$$
\displaystyle
R_{emp}[\bold{X},\bold{y},f]=\frac{1}{f}\sum^n_{i=1}l(\bold{x}^{(i)}, \bold{y}^{(i)}, f(\bold{x}^{(i)}))
$$

Note that this the sum of the loss of each output ($l(\dots)$) over each example, divided by the number of examples. In other words, $R_{emp}$ is the average loss over the training dataset. e.g. If we used MSE as the loss function, then $R_{emp}$ would be the average MSE over all examples.

Generalization error is expressed as an integral:
$$
\displaystyle
R[p,f]=E_{(x,y)\sim P}[l(\bold{x},y,f(\bold{x}))] = \int\int l(\bold{x}, y, f(\bold{x}))p(\bold{x},y) d\bold{x} dy
$$
Given that:
- $p$: joint probability density function (probability fn of seeing an example with features $x$ and true label $y$ IRL)
- $f(x)$: prediction function

Note that $E_{(x,y)\sim P}[\dots]$ means to calculate the expected value of $[\dots]$, given that $x$ and $y$ are variables that follow distribution $P$.

### Aside: The Double Integral

Note: Fubini's theorem:

$$
\displaystyle
\int\int_{x,y} f(x,y)d(x,y) = \int_x \int_y f(x,y)dydx = \int_y \int_x f(x,y)dxdy
$$

For double integrals, the order of integration generally does not matter for the result, as long as the function is "well-behaved" (i.e. if it is continuous or absolutely convergent on the region of integration). Thus, we can restructure generalization error as:

$$
\displaystyle
R[p,f]=E_{(x,y)\sim P}[l(\bold{x},y,f(\bold{x}))] = \int\int l(\bold{x}, y, f(\bold{x}))p(\bold{x},y) dy d\bold{x} 
$$

$\int \dots dy$:

For a fixed, specific input $\bold{x}$, this integral considers all possible true labels $y$ that could be associated with that $\bold{x}$. For each y, it takes the loss and weights it by the conditional probability of seeing that $y$, given the $\bold{x}$. Calculates the average error over all possible outcomes.

$\int \dots dx$:

Takes values from previous integral (expected loss for each $\bold{x}$) and sums it over all possible input values $\bold{x}$. Similar to above, the probability density function $p(x,y)$ weighs each possible $\bold{x}$ by the probability it occurs.

### Concrete example:
Let's say we have:
- $p(x)=N(0,1)$ (standard normal distribution of inputs)
- $p(y|x)=N(x,0.1)$ (labels are noisy versions of inputs)
- $l(x,y,f(x))=(y=f(x))^2$ (squared error loss)

where $N(\mu, \sigma)$ defines a normal distribution with mean $\mu$ and STD $\sigma$.

Then generalization loss becomes:

$$
\displaystyle
R[p,f]=\int p(\bold{x}) \ [ \int (y-f(x))^2 p(y|x) dy] \ d\bold{x}
$$

The inner integration gets the expected loss over all possible labels, given some $\bold{x}$. The outer integration then extends that expected loss over all possible $\bold{x}$.

Note that problematically, we can never calculate the generalization error $R$ exactly. The precise form of the density function $p(\bold{x}, y)$ is often unknown, and moreover, we cannot sample a infinite stream of data points. Thus, in practice we must estimate the generalization error using the test dataset (examples $\bold{X'}$, labels $\bold{y'}$). We apply the same formula used for calculating empirical training error to $\bold{X'}$ and $\bold{y'}$.

When evaluating our classifier on the test set, we are working with a *fixed* classifier (independent of the sample of the test set), and thus estimating error is simply a mean estimation. This does not apply to the training error, which will be a biased estimate of true error on underlying population.

Central question of generalization: when should we expect our training error to be close to the population error (and thus the generalization error)?

## Model Complexity

Simple models + Abdundant data => Training and generalization error are close

Complex models: training error can decrease, but generalization gap can grow

e.g. Imagine a model class so expressive that for any dataset of $n$ examples, we can find a set of parameters that can perfectly fit arbitrary labels, even if randomly assigned. Even if we perfectly fit training dta, we cannot conclude anything about generalization error.

When a model is capable of fitting arbitrary labels, low training error does not necessarily imply low generalization error. Neither does it imply high generalizaiton error. Examples are commonly deep neural networks, which can generalize well enough in practice but are too powerful to allow us to conclude much based on training data alone. We must rely on holdout data (i.e validation data) and *validation error*.

# 3.6.2: Underfitting or Overfitting?

Common situations to look out for:
1. Training and validation error are both large but with very little difference between them
    - Model was unable to reduce training error $\rightarrow$ Model is too simple (AKA inefficiently expressive) to capture desired pattern
    - Moreover, since generalization gap ($R_{emp}-R$) between training and generalization errors is small, we have reason to believe that we could get away with a more complex model.
    - This is **underfitting**.
1. Training error is significantly lower than validation error
    - Severe **overfitting**
    - Not always bad - deep models perform far better on training data than holdout data

Ultimately, we usually care about driving the generalization error lower, and only care about the gap insofar as it becomes an obstacle to that end.

Note that if training error is zero, then the generalization gap ($R_{emp}-R$) is precisely equal to the generalization error and we can make progress **only by reducing the gap**.

## Polynomial Curve Fitting

Given training data consisting of a singular feature $x$ and a corresponding real-valued label $y$, we try to find the polynomial of degree $d$ 
$$
\displaystyle
\hat{y}=\sum_{i=0}^{d}x^iw_i
$$
(polynomial theorem) for estimating the label $y$. This is just a linear regression problem where our features are given by the powers of $x$, the model's weights are given by $w_i$, and the bias is given by $w_0$ since $\forall x \in \mathbb{R} \ x^0=1$. As sycgm we can just use the squared error as our loss function.

A higher order polynomial fn is more complex than a lower order polynomial fn due to increased # of params. Given a fixed training dataset, higher order polynomials should always achieve <= performance relative to lower-degree polynomials. 

Whenever each example has a distinct $x$-value, a polynomial fn with degree equal to the number of data examples MINUS ONE ($N-1$) can fit the training set perfectly.
- Note: this is technically a 1:1 mapping of data to prediction. 
    - In ML, 1:1 means overfitting/memorization of training data. 
    - Mathematically, however, we have a 1:1 map from every unique input to every unique prediction.

See fig. 3.6.1: influence of model complexity on model underfitting and overfitting

### Dataset size

As indicated by fig. 3.6.1, dataset size plays a key part in model selection.

- Fewer samples => more likely to encounter overfitting
- More samples => generalization error decreases

For a fixed task and data distribution, model complexity should not increase more rapidly than the amount of data. Given more data, we can try a more complex model.

Absent sufficient data, simpler models may be more difficult to beat.

For many tasks, deep learning only outperforms linear models at the scale of 1000s of training examples.

# 3.6.3: Model Selection

- Final model should be selected after evaluating multiple models that differ in various ways (architectures, training objectives, selected features, data preprocessing, learning rates, etc.). This part is called *model selection.*
- Test data is here to keep us honest when examining training data, but be careful not to overfit on testing data, either.
- Ideally, we should only touch test data once - when assessing the very best model or when comparing a small numer of models with each other. 
- Reusing test sets for more experiments should be avoided. Recycling benchmark data over and over again can have a significan effect on algorithm development

Sol'n: splitting data into training, testing, and **validation**. 

Note that, in this novel, "test data" may truly be validation data, and as thus, accuracy reported in experiments are actually validation accuracy.

## Cross-Validation:

- Scarce training data => we may not be able to afford a proper validation set w/o cannibalizing our training and test sets.
- Sol'n: K-fold cross validation.
    - Split training data into *K* non-overlapping subsets.
    - Execute model training and validation *K* times, each time, training on *K-1* subsets and validating on the left-out subset. 
    - Training and validation errors are estimated by averaging over the results from the *K* experiments.

# Summary:

Some rules of thumb for handling deeper models (which are more susceptible to overfitting):
1. Use validation sets (or *K-fold cross-validation*) for model selection;
2. More complex models often require more data
3. Relevant notions of complexity include both number of params and range of values said params are allowed to take
4. Keeping all else equal, more data almost always leads to better generalization
5. Generalization depends on the IID assumption. If distribution shifts between training and testing periods, then we cannot say anything about generalization other than a (milder) assumption similar to IID.